# Tutorial 06 — `alexandria` Identity Gateway: Wallets (`Fafnir` & `IdentityHub`), DIDs, VCs & DCP

Alongside `alexandria-server` and `alexandria-client`, the **`cmd/alexandria`** binary provides the node's Self-Sovereign Identity (SSI) and OIDC Authentication Gateway, exposing:
- **OpenAPI 3.0** specification (`/openapi.json`, `/openapi.yaml`) and interactive **Swagger UI** (`/docs`).
- Public **DID Document** resolution (`/.well-known/did.json`).
- **Unified Wallet API (`/api/v1/wallet/*`)** over the `wallet.Wallet` port ([ADR 0007](../docs/adr/0007-unified-wallet-port-and-identityhub-adapter.md) and [ADR 0008](../docs/adr/0008-enriched-domain-models-mutation-queries-and-structured-errors.md)):
  - Asymmetric PEM and JWK key management (`/keys`, `:id/rotate` rotation, and `:id/revoke` revocation).
  - DID lifecycle and Service Endpoints (`/did`, `/did/:id/endpoints`, `/did/:id/publish`).
  - Participant contexts in **Eclipse EDC IdentityHub** (`/participants`).
  - W3C VC 1.0/2.0 Verifiable Credentials (`/credential`, `/vcs`) and **DCP** issuance requests (`/dcp/request`).
  - Structured error responses preserving the upstream provider payload (`originalResponse`).

> **Self-contained execution note:** The cell below automatically starts a local reference HTTP server implementing the exact OpenAPI contract of `internal/ssi-auth/rest/wallet_router.go`, so you can execute and experiment with every HTTP request without requiring external Docker containers.

In [1]:
import http.server
import json
import threading
import urllib.parse
from lab import free_port, http_request, repo_root

# Reference HTTP server for local Docker-free lab execution (implements wallet_router.go routes)
class _WalletLabState:
    def __init__(self) -> None:
        self.keys = {
            "key-default-1": {
                "id": "key-default-1",
                "algorithm": "EdDSA",
                "keyType": "Ed25519",
                "publicKey": "MCowBQYDK2VwAyEA...",
                "serializedPublicKey": '{"kty":"OKP","crv":"Ed25519","x":"11qYAYKxCrfVS_7TyWQHOg7hcvPapiMlrwIaaPcHURo"}',
                "keyContext": "JsonWebKey2020",
                "usage": ["sign", "verify"],
                "defaultPair": True,
                "privateKeyAlias": "key-default-1-alias",
            }
        }
        self.dids = {
            "did:web:localhost": {
                "did": "did:web:localhost",
                "default": True,
                "state": "PUBLISHED",
                "keys": [self.keys["key-default-1"]],
                "services": [],
            }
        }
        self.participants = {}
        self.credentials = []
        self.dcp_requests = {}

STATE = _WalletLabState()

class _AlexandriaHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def _send(self, code: int, body: dict | list | None = None) -> None:
        self.send_response(code)
        if body is not None:
            data = json.dumps(body).encode("utf-8")
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        else:
            self.end_headers()

    def _body(self) -> dict:
        length = int(self.headers.get("Content-Length", "0"))
        return json.loads(self.rfile.read(length).decode("utf-8")) if length > 0 else {}

    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        p = parsed.path
        q = urllib.parse.parse_qs(parsed.query)
        if p == "/healthz":
            return self._send(200, {"status": "ok"})
        if p == "/.well-known/did.json":
            return self._send(200, {
                "@context": ["https://www.w3.org/ns/did/v1"],
                "id": "did:web:localhost",
                "verificationMethod": [{"id": "did:web:localhost#key-default-1", "type": "JsonWebKey2020", "controller": "did:web:localhost"}],
                "service": STATE.dids["did:web:localhost"]["services"],
            })
        if p == "/api/v1/wallet/is-linked":
            return self._send(200)
        if p == "/api/v1/wallet/info":
            return self._send(200, {"provider": "IdentityHub", "did": "did:web:localhost", "linked": True})
        if p == "/api/v1/wallet/keys":
            return self._send(200, list(STATE.keys.values()))
        if p == "/api/v1/wallet/did":
            return self._send(200, {"did": "did:web:localhost"})
        if p == "/api/v1/wallet/did/all":
            return self._send(200, list(STATE.dids.values()))
        if p.startswith("/api/v1/wallet/did/") and p.endswith("/state"):
            did_id = urllib.parse.unquote(p[len("/api/v1/wallet/did/"):-len("/state")])
            d = STATE.dids.get(did_id)
            return self._send(200, {"did": did_id, "state": d["state"] if d else "UNKNOWN"})
        if p == "/api/v1/wallet/vcs":
            vc_type = q.get("type", [None])[0]
            creds = STATE.credentials
            if vc_type:
                creds = [c for c in creds if vc_type in c.get("types", [])]
            return self._send(200, creds)
        if p.startswith("/api/v1/wallet/dcp/request/"):
            req_id = p.split("/")[-1]
            return self._send(200, STATE.dcp_requests.get(req_id, {"requestId": req_id, "status": "RECEIVED"}))
        if p.startswith("/api/v1/wallet/participants/"):
            pid = p.split("/")[-1]
            return self._send(200, STATE.participants[pid])
        return self._send(404, {"error": "not found"})

    def do_POST(self):
        p = urllib.parse.urlparse(self.path).path
        b = self._body()
        if p == "/api/v1/wallet/link":
            return self._send(200, STATE.dids["did:web:localhost"])
        if p == "/api/v1/wallet/keys":
            kid = b.get("id", "key-new")
            key_obj = {
                "id": kid,
                "algorithm": "EdDSA",
                "keyType": b.get("keyType", "Ed25519"),
                "serializedPublicKey": '{"kty":"OKP","crv":"Ed25519","x":"new_jwk_public_key_bytes"}',
                "keyContext": "JsonWebKey2020",
                "usage": ["sign", "verify"],
                "defaultPair": False,
                "privateKeyAlias": b.get("alias", f"{kid}-alias"),
            }
            STATE.keys[kid] = key_obj
            return self._send(201, key_obj)
        if p.startswith("/api/v1/wallet/keys/") and p.endswith("/rotate"):
            kid = p.split("/")[-2]
            k = STATE.keys[kid]
            k["privateKeyAlias"] = f"{kid}-rotated"
            return self._send(200, k)
        if p.startswith("/api/v1/wallet/did/") and p.endswith("/endpoints"):
            did_id = urllib.parse.unquote(p[len("/api/v1/wallet/did/"):-len("/endpoints")])
            d = STATE.dids[did_id]
            if any(s["id"] == b["id"] for s in d["services"]):
                return self._send(409, {
                    "error": "conflict: resource already exists",
                    "originalResponse": {
                        "provider": "identityhub",
                        "statusCode": 409,
                        "path": "/api/identity/v1alpha/participants/ZGlkOndlYjpsb2NhbGhvc3Q/dids/endpoints",
                        "body": [{"message": f"Service endpoint '{b['id']}' already exists on DID '{did_id}'", "type": "ObjectConflict"}],
                    },
                })
            d["services"].append({"id": b["id"], "type": b["type"], "serviceEndpoint": b["url"]})
            return self._send(201, d)
        if p.startswith("/api/v1/wallet/did/") and p.endswith("/publish"):
            did_id = urllib.parse.unquote(p[len("/api/v1/wallet/did/"):-len("/publish")])
            STATE.dids[did_id]["state"] = "PUBLISHED"
            return self._send(200, {"did": did_id, "state": "PUBLISHED"})
        if p == "/api/v1/wallet/participants":
            pid = b["id"]
            part = {"id": pid, "did": b["did"], "active": b.get("active", True), "apiToken": "tok_ihub_generated_9f8a7b"}
            STATE.participants[pid] = part
            return self._send(201, part)
        if p == "/api/v1/wallet/credential":
            cred = {
                "id": b.get("id", "vc-1"),
                "issuer": "did:web:issuer.example.com",
                "subject": "did:web:mobility.example.com",
                "types": ["VerifiableCredential", "DataSpaceParticipant"],
                "format": b.get("format", "VC1_0_JWT"),
                "participantContextId": b.get("participantContextId", "participant-mobility-1"),
                "issuanceDate": "2026-09-09T10:00:00Z",
                "validUntil": "2027-09-09T10:00:00Z",
                "rawVc": b.get("rawVc", "eyJhbGciOiJFZERTQSJ9..."),
            }
            STATE.credentials.append(cred)
            return self._send(201, cred)
        if p == "/api/v1/wallet/dcp/request":
            req_id = f"dcp-{b.get('holderPid', '001')}"
            STATE.dcp_requests[req_id] = {"requestId": req_id, "status": "ISSUED"}
            return self._send(202, {"requestId": req_id})
        return self._send(404, {"error": "not found"})

port = free_port()
srv = http.server.HTTPServer(("127.0.0.1", port), _AlexandriaHandler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
BASE_URL = f"http://127.0.0.1:{port}"
print(f"✔ Alexandria Wallet API endpoint active at {BASE_URL}")

✔ Alexandria Wallet API endpoint active at http://127.0.0.1:62811


## 1. Health Check, Wallet Linking, and Provider Info

Let's inspect `/healthz`, `GET /api/v1/wallet/is-linked`, `POST /api/v1/wallet/link`, and `GET /api/v1/wallet/info`:

In [2]:
http_request("GET", f"{BASE_URL}/healthz")
http_request("GET", f"{BASE_URL}/api/v1/wallet/is-linked")
http_request("POST", f"{BASE_URL}/api/v1/wallet/link")
http_request("GET", f"{BASE_URL}/api/v1/wallet/info")

content-length,16
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,440
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,71
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


## 2. Asymmetric Key Lifecycle (`POST /api/v1/wallet/keys` & `:rotate`)

Thanks to the **Mutation-and-Query** pattern ([ADR 0008](../docs/adr/0008-enriched-domain-models-mutation-queries-and-structured-errors.md)), registering or rotating a key immediately returns the enriched domain entity (`serializedPublicKey` in JWK/PEM, `keyContext`, `usage`, `privateKeyAlias`):

In [3]:
print("1) Register a new Ed25519 key (POST /api/v1/wallet/keys):")
st_k, _, key_created = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/keys",
    body={"id": "signing-key-2026", "alias": "primary-ed25519", "keyType": "Ed25519"},
)
assert st_k == 201

print("2) Rotate key with a 24h overlap window (POST /api/v1/wallet/keys/signing-key-2026/rotate):")
st_rot, _, _ = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/keys/signing-key-2026/rotate",
    body={"duration": "24h"},
)
assert st_rot == 200

print("3) List all wallet keys (GET /api/v1/wallet/keys):")
http_request("GET", f"{BASE_URL}/api/v1/wallet/keys")

1) Register a new Ed25519 key (POST /api/v1/wallet/keys):
2) Rotate key with a 24h overlap window (POST /api/v1/wallet/keys/signing-key-2026/rotate):
3) List all wallet keys (GET /api/v1/wallet/keys):


content-length,290
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,299
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,648
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


## 3. DID Management, Service Endpoints, and Structured Errors (`originalResponse`)

We add a `CredentialService` endpoint to the DID Document (`POST /api/v1/wallet/did/:id/endpoints`), publish it (`:publish`), and test registering a duplicate endpoint (receiving IdentityHub's `originalResponse` field):

In [4]:
print("1) Add Service Endpoint to DID Document (POST /api/v1/wallet/did/did:web:localhost/endpoints):")
st_ep, _, _ = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/did/did:web:localhost/endpoints",
    body={
        "id": "credential-service-1",
        "type": "CredentialService",
        "url": "https://alexandria.127.0.0.1.nip.io:8443/api/v1/dsp",
    },
)
assert st_ep == 201

print("2) Attempt to register the same duplicate Service Endpoint (expected 409 with IdentityHub originalResponse):")
st_dup, _, err_dup = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/did/did:web:localhost/endpoints",
    body={
        "id": "credential-service-1",
        "type": "CredentialService",
        "url": "https://alexandria.127.0.0.1.nip.io:8443/api/v1/dsp",
    },
)
assert st_dup == 409
assert "originalResponse" in err_dup

print("3) Publish DID and verify /.well-known/did.json:")
http_request("POST", f"{BASE_URL}/api/v1/wallet/did/did:web:localhost/publish")
http_request("GET", f"{BASE_URL}/.well-known/did.json")

1) Add Service Endpoint to DID Document (POST /api/v1/wallet/did/did:web:localhost/endpoints):
2) Attempt to register the same duplicate Service Endpoint (expected 409 with IdentityHub originalResponse):
3) Publish DID and verify /.well-known/did.json:


content-length,573
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,333
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,50
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,349
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


## 4. IdentityHub Participants, Verifiable Credentials, and DCP Requests

Finally, we create a participant in IdentityHub (`POST /api/v1/wallet/participants`), store a verifiable credential (`POST /api/v1/wallet/credential`), filter credentials by type (`GET /api/v1/wallet/vcs?type=DataSpaceParticipant`), and trigger a **DCP** credential request (`POST /api/v1/wallet/dcp/request`):

In [5]:
print("1) Create participant in IdentityHub (POST /api/v1/wallet/participants):")
st_p, _, _ = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/participants",
    body={
        "id": "participant-mobility-1",
        "did": "did:web:mobility.example.com",
        "active": True,
        "keyDescriptor": {"keyId": "key-1", "type": "Ed25519", "privateKeyAlias": "mobility-key-1"},
    },
)
assert st_p == 201

print("2) Store normalized Verifiable Credential (POST /api/v1/wallet/credential):")
st_vc, _, _ = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/credential",
    body={
        "id": "vc-membership-001",
        "participantContextId": "participant-mobility-1",
        "format": "VC1_0_JWT",
        "rawVc": "eyJhbGciOiJFZERTQSIsInR5cCI6IkpXVCJ9...",
    },
)
assert st_vc == 201

print("3) Query credentials filtered by type (?type=DataSpaceParticipant):")
st_vcs, _, vcs_list = http_request("GET", f"{BASE_URL}/api/v1/wallet/vcs?type=DataSpaceParticipant")
assert st_vcs == 200 and len(vcs_list) == 1

print("4) Request credential issuance via DCP (POST /api/v1/wallet/dcp/request):")
st_dcp, _, dcp_resp = http_request(
    "POST",
    f"{BASE_URL}/api/v1/wallet/dcp/request",
    body={
        "issuerDid": "did:web:issuer.example.com",
        "holderPid": "req-001",
        "credentials": [{"id": "cred-1", "format": "VC1_0_JWT", "type": "MembershipCredential"}],
    },
)
assert st_dcp == 202
req_id = dcp_resp["requestId"]

print("5) Check DCP request status (GET /api/v1/wallet/dcp/request/:id):")
http_request("GET", f"{BASE_URL}/api/v1/wallet/dcp/request/{req_id}")
srv.shutdown()

1) Create participant in IdentityHub (POST /api/v1/wallet/participants):
2) Store normalized Verifiable Credential (POST /api/v1/wallet/credential):
3) Query credentials filtered by type (?type=DataSpaceParticipant):
4) Request credential issuance via DCP (POST /api/v1/wallet/dcp/request):
5) Check DCP request status (GET /api/v1/wallet/dcp/request/:id):


content-length,128
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,372
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,374
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,28
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11


content-length,48
content-type,application/json; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:57 GMT"
server,BaseHTTP/0.6 Python/3.10.11
